---
# Методы статистической обработки информации
---
## Лабораторная работа 2 / Регрессионный анализ / Вариант 15
---

---
## Подготовка
---

### Импорт библиотек

In [141]:
from math             import log10
from MoSDP_data       import get_data
from MoSDP_operations import *

### Получение данных варианта

In [142]:
variant  = 15 # int(input("Введите номер варианта: "))
var_data = get_data(variant)

### Подготовка данных


In [143]:
Y_linear = transpose([var_data[0]])
X_linear = transpose(var_data[1:])

n = len(X_linear)

for x in X_linear:
    x.insert(0, 1)

numbers_linear = [i for i in range(len(X_linear[0]))]
numbers_pow    = [i for i in range(len(X_linear[0]))]

### Приложение

In [144]:
# По "таблице 2": a = 0.05 и v = n - k - 1 = 53 - 5 - 1 = 47
t_table = {(0.05, 40): 2.021, (0.1, 40): 1.684, (0.2, 40): 1.303, (0.3, 40): 1.050 , (0.4, 40): 0.851}

---
## Задание 1
---

### Функция "Оценка уравнения регрессии"

In [145]:
def estimate(Y, X):
    xTx = matrix_multiplication(transpose(X), X)
    xTy = matrix_multiplication(transpose(X), Y)
    b   = matrix_multiplication(inverse_matrix(xTx), xTy)

    return b

### Функция "Формирование уравнения"

In [146]:
def draw(b, numbers, form):
    equation = f"y = {b[0][0]}"

    for i in range(1, len(b)):
        if form == "linear":
            if b[i][0] > 0:
                equation += f" + {b[i][0]}"
            else:
                equation += f" - {abs(b[i][0])}"
            equation += f" * x{numbers[i]}"
        else:
            equation += f" * x{numbers[i]}^({b[i][0]})"

    print(equation)

### Оценка линейного уравнения регрессии

In [147]:
b_linear = estimate(Y_linear, X_linear)
print("b_linear:", b_linear, "\n")

draw(b_linear, numbers_linear, "linear")

b_linear: [[-12.560998303339147], [18.21127665232234], [1.6273983943464145], [0.5084514879716515], [0.2785008638589499], [0.21507667359831828]] 

y = -12.560998303339147 + 18.21127665232234 * x1 + 1.6273983943464145 * x2 + 0.5084514879716515 * x3 + 0.2785008638589499 * x4 + 0.21507667359831828 * x5


### Оценка степенного уравнения регрессии

In [148]:
Y_pow = copy(Y_linear)
X_pow = copy(X_linear)

for i in range(len(X_linear)):
    Y_pow[i][0] = log10(Y_pow[i][0])

    for j in range(1, len(X_linear[0])):
        X_pow[i][j] = log10(X_pow[i][j])

b_lg        = estimate(Y_pow, X_pow)
b_pow       = copy(b_lg)
b_pow[0][0] = 10 ** b_pow[0][0]
print("b_pow:", b_pow, "\n")

draw(b_pow, numbers_pow, "pow")

b_pow: [[0.5749714020371827], [3.591015023519958], [-0.027089077716006216], [0.24718836073139272], [0.2095603646767561], [1.0877541678012221]] 

y = 0.5749714020371827 * x1^(3.591015023519958) * x2^(-0.027089077716006216) * x3^(0.24718836073139272) * x4^(0.2095603646767561) * x5^(1.0877541678012221)


---
## Задание 2
---

### Функция "Значимость уравнения"

In [149]:
def significance_equation(Y, X, b):
    Y_reg = matrix_multiplication(X, b)

    Q = 0
    for i in range(len(Y)):
        Q += (Y[i][0] - Y_reg[i][0]) ** 2

    k   = len(X[0]) - 1
    S_2 = Q / (n - k - 1)

    Q_r = matrix_multiplication(transpose(Y_reg), Y_reg)

    F_critical = 2.25 # По "таблице 4": a = 0.05; v1 = k + 1 = 5 + 1 = 6 и v2 = n - k - 1 = 53 - 5 - 1 = 47
    F          = (Q_r[0][0] / (k + 1)) / S_2

    res = ('<=', 'не ')
    if F > F_critical:
        res = ('>', '')

    print(f"F_набл = {F} {res[0]} {F_critical}: {res[1]}значимое")

    return S_2

### Значимость линейного уравнения

In [150]:
S_2_linear = significance_equation(Y_linear, X_linear, b_linear)

F_набл = 83.74135090633733 > 2.25: значимое


### Значимость степенного уравнения

In [151]:
S_2_pow = significance_equation(Y_pow, X_pow, b_lg)

F_набл = 434.62842715579666 > 2.25: значимое


### Функция "Значимость коэффициентов регрессии"

In [152]:
def significance_coefficients(X, b, S_2, t_critical):
    xTx = inverse_matrix(matrix_multiplication(transpose(X), X))
    S   = matrix_by_a_number(xTx, S_2)

    t = []
    for i in range(1, len(S)):
        t.append(b[i][0] / (S[i][i] ** 0.5))

        res = (' <=', 'не')
        if abs(t[-1]) > t_critical:
            res = (' > ', '  ')

        print(f"t_набл(b{i}) = |{t[-1]:{7}.3f}| {res[0]} {t_critical}: {res[1]} значимый")

    return t

### Значимость коэффициентов регрессии линейного уравнения

In [153]:
t_linear = significance_coefficients(X_linear, b_linear, S_2_linear, t_table[(0.05, 40)])

t_набл(b1) = |  0.860|  <= 2.021: не значимый
t_набл(b2) = |  0.598|  <= 2.021: не значимый
t_набл(b3) = |  0.428|  <= 2.021: не значимый
t_набл(b4) = |  1.457|  <= 2.021: не значимый
t_набл(b5) = |  0.906|  <= 2.021: не значимый


### Значимость коэффициентов регрессии степенного уравнения

In [154]:
t_pow = significance_coefficients(X_pow, b_pow, S_2_pow, t_table[(0.05, 40)])

t_набл(b1) = |  2.524|  >  2.021:    значимый
t_набл(b2) = | -0.438|  <= 2.021: не значимый
t_набл(b3) = |  1.420|  <= 2.021: не значимый
t_набл(b4) = |  1.818|  <= 2.021: не значимый
t_набл(b5) = |  2.571|  >  2.021:    значимый


---
## Задание 3
---

### Функция "Мультиколлинеарность"

In [173]:
def multicollinearity(sample):
    print("Матрица парных коэффициентов корреляции:")
    R = pairwise_correlation_coefficients(sample)
    print()

"""
related = []
for i in range(1, len(R)):
    for j in range(i + 1, len(R)):
        if abs(R[i][j]) > 0.8:
            print(f"X{i} и X{j} - тесно связанные: r_{i}{j} = |{R[i][j]:.3f}| > 0.8")

            related.append(i)
            related.append(j)

X_without_related_linear = copy(X_linear)
X_without_related_pow    = copy(X_pow)

if len(related) > 0:
    i = list(set(related))[0]

    max_correlation = 0
    for j in related[1:]:
        if max_correlation < abs(R[0][j]):
            max_correlation = abs(R[0][j])
            i               = j

    print(f"\nОставляем X{i} из X{", X".join(str(j) for j in related)}")
    related.remove(i)

    for j in range(len(X_linear)):
        for i in related:
            X_without_related_linear[j].pop(i)
            X_without_related_pow[j].pop(i)

else:
    print("Нет тесно связанных X")
"""

'\nrelated = []\nfor i in range(1, len(R)):\n    for j in range(i + 1, len(R)):\n        if abs(R[i][j]) > 0.8:\n            print(f"X{i} и X{j} - тесно связанные: r_{i}{j} = |{R[i][j]:.3f}| > 0.8")\n\n            related.append(i)\n            related.append(j)\n\nX_without_related_linear = copy(X_linear)\nX_without_related_pow    = copy(X_pow)\n\nif len(related) > 0:\n    i = list(set(related))[0]\n\n    max_correlation = 0\n    for j in related[1:]:\n        if max_correlation < abs(R[0][j]):\n            max_correlation = abs(R[0][j])\n            i               = j\n\n    print(f"\nОставляем X{i} из X{", X".join(str(j) for j in related)}")\n    related.remove(i)\n\n    for j in range(len(X_linear)):\n        for i in related:\n            X_without_related_linear[j].pop(i)\n            X_without_related_pow[j].pop(i)\n\nelse:\n    print("Нет тесно связанных X")\n'

### Мультиколлинеарность линейного уравнения

In [177]:
multicollinearity(var_data)

Матрица парных коэффициентов корреляции:
[1                        | 0.054840792487636555     | 0.15115051013848507      | 0.0003555860122828476    | 0.23405035667830304      | 0.018075047394182264     ]
[0.054840792487636555     | 1                        | -0.06874216425985696     | -0.016615678293707227    | 0.2193289370025393       | -0.9402329788072865      ]
[0.15115051013848507      | -0.06874216425985696     | 1                        | 0.4333027759567459       | -0.04467957562887469     | 0.14675271575071883      ]
[0.0003555860122828476    | -0.016615678293707227    | 0.4333027759567459       | 1                        | -0.5084487611839539      | 0.03736158062663443      ]
[0.23405035667830304      | 0.2193289370025393       | -0.04467957562887469     | -0.5084487611839539      | 1                        | -0.14710495223940964     ]
[0.018075047394182264     | -0.9402329788072865      | 0.14675271575071883      | 0.03736158062663443      | -0.14710495223940964     | 1       

### Мультиколлинеарность степенного уравнения

In [176]:
var_data_lg = copy(var_data)

for row in var_data_lg:
    for i in range(len(row)):
        row[i] = log10(row[i])

multicollinearity(var_data_lg)

Матрица парных коэффициентов корреляции:
[1                        | 0.10259764042570861      | 0.1327619563703737       | 0.07812534391931236      | 0.21797415230395475      | 0.05768242341919306      ]
[0.10259764042570861      | 1                        | -0.09311330813646256     | -0.006143617306348758    | 0.19597908620142443      | -0.9100093133233252      ]
[0.1327619563703737       | -0.09311330813646256     | 1                        | 0.4025979614542848       | -0.0912299433840107      | 0.2406051574322556       ]
[0.07812534391931236      | -0.006143617306348758    | 0.4025979614542848       | 1                        | -0.4446836830988275      | 0.002790920878960479     ]
[0.21797415230395475      | 0.19597908620142443      | -0.0912299433840107      | -0.4446836830988275      | 1                        | -0.14163967790330317     ]
[0.05768242341919306      | -0.9100093133233252      | 0.2406051574322556       | 0.002790920878960479     | -0.14163967790330317     | 1       

### Функция "Алгоритм пошагового регрессионного анализа"

In [159]:
def regression_analysis(Y, X, numbers, form):
    for parameters in t_table:
        t_critical = t_table[parameters]

        print("=" * 97)
        print(f"При a = {parameters[0]}")
        print("=" * 97, "\n")



        new_X       = copy(X)
        new_numbers = numbers.copy()

        step = 0
        while min(new_t) <= t_critical and len(new_t) > 1:
            step += 1

            print(f"{"-" * 45} Шаг {step} {"-" * 45}")

            j = new_t.index(min(new_t)) + 1
            print(f"Минимальный t: |t_набл(b{j})| <= {t_critical}")

            for x in new_X:
                x.pop(j)

            new_numbers.pop(j)

            b = estimate(Y, new_X)
            print(f"\nНовая оценка b_{form}: {b}")

            draw(b, new_numbers, form)
            print()

            S_2   = significance_equation(Y, new_X, b)
            new_t = significance_coefficients(new_X, b, S_2, t_critical)
            new_t = [abs(i) for i in new_t]
            print()

        if min(new_t) > t_critical:
            print("=" * 97)
            print(f"b_{form}: {b}")
            print()
            draw(b, new_numbers, form)
            print("=" * 97)

            return new_X, b, new_numbers

    return None, None, None

### Пошаговый регрессионный анализ линейного уравнения

In [157]:
new_X_linear, b_linear, new_numbers_linear = regression_analysis(Y_linear, X_without_related_linear, numbers_linear, "linear")

При a = 0.05

--------------------------------------------- Шаг 1 ---------------------------------------------
Минимальный t: |t_набл(b3)| <= 2.021

Новая оценка b_linear: [[5.092040451995965], [0.6748677623711927], [2.7370930326361744], [0.26097327685662997]]
y = 5.092040451995965 + 0.6748677623711927 * x1 + 2.7370930326361744 * x2 + 0.26097327685662997 * x4

F_набл = 127.77550042402119 > 2.25: значимое
t_набл(b1) = |  0.098|  <= 2.021: не значимый
t_набл(b2) = |  1.185|  <= 2.021: не значимый
t_набл(b3) = |  1.697|  <= 2.021: не значимый

--------------------------------------------- Шаг 2 ---------------------------------------------
Минимальный t: |t_набл(b1)| <= 2.021

Новая оценка b_linear: [[5.573010185006879], [2.7234238508757462], [0.26423889093012676]]
y = 5.573010185006879 + 2.7234238508757462 * x2 + 0.26423889093012676 * x4

F_набл = 173.80698785215486 > 2.25: значимое
t_набл(b1) = |  1.193|  <= 2.021: не значимый
t_набл(b2) = |  1.778|  <= 2.021: не значимый

------------

### Пошаговый регрессионный анализ степенного уравнения

In [158]:
new_X_pow, b_pow, new_numbers_pow = regression_analysis(Y_pow, X_without_related_pow, numbers_pow, "pow")

При a = 0.05

--------------------------------------------- Шаг 1 ---------------------------------------------
Минимальный t: |t_набл(b2)| <= 2.021

Новая оценка b_pow: [[0.6938412897802024], [0.18673788021083482], [0.2297334303163261], [0.23384606383432782]]
y = 0.6938412897802024 * x1^(0.18673788021083482) * x3^(0.2297334303163261) * x4^(0.23384606383432782)

F_набл = 589.3181416211814 > 2.25: значимое
t_набл(b1) = |  0.317|  <= 2.021: не значимый
t_набл(b2) = |  1.397|  <= 2.021: не значимый
t_набл(b3) = |  1.951|  <= 2.021: не значимый

--------------------------------------------- Шаг 2 ---------------------------------------------
Минимальный t: |t_набл(b1)| <= 2.021

Новая оценка b_pow: [[0.6618282281231735], [0.23453761828713837], [0.24203893885754013]]
y = 0.6618282281231735 * x3^(0.23453761828713837) * x4^(0.24203893885754013)

F_набл = 800.121015147127 > 2.25: значимое
t_набл(b1) = |  1.445|  <= 2.021: не значимый
t_набл(b2) = |  2.086|  >  2.021:    значимый

-------------